# 05 - Phân tích lỗi

In [ ]:
import re, sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

from src.config import (LABELS, DATA_DIR, MODELS_DIR, RESULTS_METRICS, RESULTS_FIGURES, RESULTS_PREDICTIONS, PREPROCESS_PRESETS, text_col)
print("ROOT =", ROOT)

In [ ]:
sys.path.insert(0, str(ROOT / "src"))
pd.set_option("display.max_colwidth", 200)   # đọc được nguyên câu lỗi
from preprocess import TEENCODE, EDGE_PUNCT, EMOJI_RE, HOMOGLYPH   # từ điển/regex dùng chung với bước tiền xử lý

CANDIDATES = ["baseline", "phobert", "visobert"]
MODELS = [m for m in CANDIDATES if (RESULTS_PREDICTIONS / f"{m}_test.csv").exists()]
print("Mô hình đã có dự đoán:", MODELS)
assert MODELS, "Chưa có file dự đoán nào trong results/predictions/"

test = pd.read_csv(DATA_DIR / "test.csv")
P = {}
for m in MODELS:
    p = pd.read_csv(RESULTS_PREDICTIONS / f"{m}_test.csv")
    
    # file dự đoán chỉ lưu văn bản đã xử lý (không có id/câu gốc) -> ghép theo thứ tự dòng và kiểm tra khớp trước khi dùng
    assert len(p) == len(test), f"{m}: số dòng dự đoán ({len(p)}) khác test.csv ({len(test)}) - có thể đã chạy với --limit"
    assert (p["true_id"].values == test["label_id"].values).all(), f"{m}: nhãn thật không khớp test.csv"
    assert (p["text"].fillna("").values == test[text_col(m)].fillna("").values).all(), f"{m}: văn bản không khớp text_{m}"
    
    # file dự đoán mới đã có sẵn id/free_text; file cũ thì lấy từ test.csv (đã kiểm khớp ở trên)
    if "id" not in p.columns:
        p["id"] = test["id"].values
    if "free_text" not in p.columns:
        p["free_text"] = test["free_text"].values
    assert (p["id"].values == test["id"].values).all(), f"{m}: id không khớp test.csv"
    P[m] = p
print("Đã ghép xong, thứ tự dòng khớp với test.csv.")

## 1. So sánh các mô hình

In [ ]:
from src.compare import build_summary
display(build_summary())  

## 2. Lỗi theo cặp nhãn (thật - đoán)

In [ ]:
pairs_dict = {}
for m in MODELS:
    # Lọc ra các câu dự đoán sai và đếm số lượng theo cặp (true, pred)
    incorrect_df = P[m][~P[m].correct]
    pairs_dict[m] = incorrect_df.groupby(["true_label", "pred_label"]).size()

pairs = pd.DataFrame(pairs_dict).fillna(0).astype(int)
display(pairs)

total_errors = {}
for m in MODELS:
    total_errors[m] = int((~P[m].correct).sum())

print("Tổng số lỗi:", total_errors, "trên", len(test), "câu")

## 3. Lỗi và độ tin cậy

In [ ]:
rows = []
for m, p in P.items():
    err = p[~p.correct]
    rows.append({"mô hình": m, "số lỗi": len(err),
                 "conf TB (đúng)": round(p[p.correct].confidence.mean(), 3),
                 "conf TB (sai)": round(err.confidence.mean(), 3),
                 "lỗi có conf > 0.8": int((err.confidence > 0.8).sum()),
                 "% lỗi có conf > 0.8": round((err.confidence > 0.8).mean() * 100, 1)})
display(pd.DataFrame(rows))

## 4. Xem mẫu lỗi
Mỗi cặp nhãn lỗi nhiều nhất hiển thị `N_SAMPLE` câu ngẫu nhiên (seed cố định)

In [ ]:
FOCUS = MODELS[-1]    
N_SAMPLE = 6

err = P[FOCUS][~P[FOCUS].correct]
top_pairs = err.groupby(["true_label", "pred_label"]).size().sort_values(ascending=False).index[:4]
for t, pr in top_pairs:
    sub = err[(err.true_label == t) & (err.pred_label == pr)]
    print(f"\n  [{FOCUS}] Thật: {t}  ->  Đoán: {pr}  ({len(sub)} câu)")
    display(sub.sample(min(N_SAMPLE, len(sub)), random_state=42)[["free_text", "confidence"]])

In [ ]:
# câu tất cả mô hình cùng đoán sai: thường là nhãn nhiễu hoặc câu thật sự khó
if len(MODELS) > 1:
    wrong_all = np.logical_and.reduce([~P[m].correct.values for m in MODELS])
    print(f"{int(wrong_all.sum())} câu cả {len(MODELS)} mô hình cùng sai")
    shown = P[FOCUS][wrong_all][["free_text", "true_label", "pred_label", "confidence"]]
    display(shown.sample(min(15, len(shown)), random_state=42))
else:
    print("Mới có 1 mô hình - ô này sẽ có nội dung khi có thêm PhoBERT/ViSoBERT.")

## 5. Phân tích

### 5.1. Mất cân bằng dữ liệu

In [ ]:
rows = []
for m in MODELS:
    pc = json.load(open(RESULTS_METRICS / f"{m}.json", encoding="utf-8"))["test"]["per_class"]
    for cname, v in pc.items():
        rows.append({"mô hình": m, "lớp": cname, **v})
imb = pd.DataFrame(rows)
imb["% test"] = (imb.support / imb.groupby("mô hình").support.transform("sum") * 100).round(1)
display(imb.round(4))
print("F1 theo lớp:")
display(imb.pivot(index="lớp", columns="mô hình", values="f1").reindex(list(LABELS.values())).round(4))

### 5.2. Bias đối với các từ chỉ nhóm người hoặc danh tính
- False positive (FP): câu bình thường bị đoán nhầm thành độc hại
- Các câu BÌNH THƯỜNG có chứa từ chỉ nhóm, tính tỉ lệ đoán nhầm là độc hại (FPR) và so với FPR chung

In [ ]:
IDENTITY_TERMS = {
    "chính trị": ["cộng sản", "đảng", "phản động", "dân chủ"],
    "tôn giáo": ["công giáo", "thiên chúa", "phật giáo", "hồi giáo", "tin lành"],
    "vùng miền": ["bắc kỳ", "nam kỳ", "miền bắc", "miền nam", "miền trung"],
    "quốc tịch": ["trung quốc", "hàn quốc", "nhật bản", "campuchia", "thái lan", "người mỹ"],
    "giới / xu hướng": ["gay", "đồng tính", "lgbt", "phụ nữ", "đàn bà", "đàn ông"],
}


def has_term(series, terms):
    pat = "|".join(rf"(?<!\w){re.escape(t)}(?!\w)" for t in terms)
    return series.fillna("").str.lower().str.contains(pat, regex=True)


rows = []
for m, p in P.items():
    normal = p[p.true_id == 0]
    base_fpr = (normal.pred_id != 0).mean() * 100
    for g, terms in IDENTITY_TERMS.items():
        mask = has_term(p.free_text, terms)
        sub = p[mask]
        sub_n = sub[sub.true_id == 0]
        rows.append({"mô hình": m, "nhóm": g, "số câu": int(mask.sum()),
                     "% nhãn độc hại thật": round((sub.true_id != 0).mean() * 100, 1) if len(sub) else np.nan,
                     "số câu BÌNH THƯỜNG": len(sub_n),
                     "FPR nhóm (%)": round((sub_n.pred_id != 0).mean() * 100, 1) if len(sub_n) else np.nan,
                     "FPR chung (%)": round(base_fpr, 1)})
display(pd.DataFrame(rows))

### 5.3. Viết tắt, nói lái, chèn ký tự

In [ ]:
VN_DIACRITIC = re.compile(r"[àáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ]", re.I)

def flags(text):
    text = str(text)
    toks = [t.strip(EDGE_PUNCT).lower() for t in text.split()]
    return pd.Series({
        "viết tắt/teencode": any(t in TEENCODE for t in toks),
        "chèn ký tự (vd. đ.m)": bool(re.search(r"(?<!\w)[^\W\d_](?:[.\-_*][^\W\d_])+(?!\w)", text)),
        "không dấu (từ 4 từ trở lên)": len(text.split()) >= 4 and not VN_DIACRITIC.search(text),
        "trích dẫn (ngoặc kép)": bool(re.search(r"[\"“”«»]", text)),
        "dẫn lời (bảo/nói là)": bool(re.search(r"(?<!\w)(?:bảo|nói|chửi|gọi|hỏi) (?:là|rằng)(?!\w)", text.lower())),
    })


F = test["free_text"].fillna("").apply(flags)
OBF_COLS = ["viết tắt/teencode", "chèn ký tự (vd. đ.m)", "không dấu (từ 4 từ trở lên)"]
QUOTE_COLS = ["trích dẫn (ngoặc kép)", "dẫn lời (bảo/nói là)"]


def error_by_flag(cols):
    rows = []
    for m, p in P.items():
        for f in cols:
            mk = F[f].values
            rows.append({"mô hình": m, "đặc điểm": f, "số câu có": int(mk.sum()),
                         "% lỗi khi có": round((~p.correct[mk]).mean() * 100, 1) if mk.any() else np.nan,
                         "% lỗi khi không": round((~p.correct[~mk]).mean() * 100, 1)})
    return pd.DataFrame(rows)


display(error_by_flag(OBF_COLS))

### 5.4. Trích dẫn nội dung độc hại và công kích trực tiếp

In [ ]:
display(error_by_flag(QUOTE_COLS))

quoted = F[QUOTE_COLS].any(axis=1).values
q_err = P[FOCUS][quoted & ~P[FOCUS].correct.values]
print(f"[{FOCUS}] {len(q_err)} câu lỗi có trích dẫn/dẫn lời (trên {int(quoted.sum())} câu có đặc điểm này)")
display(q_err[["free_text", "true_label", "pred_label", "confidence"]].head(20))

In [ ]:
# xuất toàn bộ câu lỗi của FOCUS ra CSV
out = P[FOCUS][~P[FOCUS].correct][["id", "free_text", "true_label", "pred_label", "confidence"]].join(F).assign(ghi_chu="")
out_path = ROOT / "results" / f"error_cases_{FOCUS}.csv"
out.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"Đã lưu {len(out)} câu lỗi tại {out_path.relative_to(ROOT)}")